# Contrastive knee MRI submission
Attach the competition and the private exported `bundle/` dataset. Enable a GPU and disable Internet. The bundle includes complete image weights and label heads. No training data or reports are needed.
The runtime needs the dependencies in `requirements-kaggle.txt`; if the Kaggle image differs, attach compatible Linux wheels as a private dataset and install with `pip --no-index --find-links`. Save a version and submit its `/kaggle/working/submission.csv` output.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import torch

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
# Attach the competition and a PRIVATE dataset containing the exported bundle/.
# Set BUNDLE/DATA_ROOT explicitly if multiple matching datasets are attached.
bundles = sorted(Path("/kaggle/input").rglob("bundle.json"))
assert len(bundles) == 1, f"Expected one bundle.json; found {bundles}"
BUNDLE = bundles[0].parent
runtime_source = BUNDLE / "src"
runtime_archive = BUNDLE / "src.zip"
if not runtime_source.is_dir():
    assert runtime_archive.is_file(), f"Missing runtime source: {runtime_source} or {runtime_archive}"
    os.environ["PYTHONPATH"] = str(runtime_archive)
roots = sorted({p.parent for p in Path("/kaggle/input").rglob("sample_submission.csv")
                if (p.parent / "test_series.csv").is_file()})
assert len(roots) == 1, f"Expected one competition data root; found {roots}"
DATA_ROOT = roots[0]
subprocess.run([
    sys.executable, str(BUNDLE / "predict_kaggle.py"),
    "--bundle", str(BUNDLE), "--data-root", str(DATA_ROOT),
    "--output", "/kaggle/working/submission.csv",
    "--device", "cuda" if torch.cuda.is_available() else "cpu", "--batch-size", "8",
], check=True)
